# Reducción de varianza en Monte Carlo

**Clase del 19 de octubre de 2026**

Los tres ejercicios de este cuaderno parten de la misma pregunta: ¿cómo conseguir una estimación Monte Carlo más precisa sin simular más trayectorias? Las partes 1 y 2 son ejercicios previos y la parte 3 es el ejercicio de variable de control.

| Parte | Tema | Técnica |
|:--:|:--|:--|
| 1 | Media de los valores negativos y positivos de una normal estándar | Simetría |
| 2 | Call europea con los valores Z dados por el profesor | Variables antitéticas |
| 3 | Call asiática aritmética con 100 rutas | Variable de control geométrica |

Al final del cuaderno hay un diccionario con los términos que aparecen.

---

## Parte 1. Simetría de la normal estándar

Se simulan 10,000 valores de $Z \sim N(0,1)$ y se calcula la media de los que salieron negativos y la de los que salieron positivos.

**Valor teórico.** Para la mitad positiva de una normal estándar:

$$E[Z \mid Z>0] = \frac{\varphi(0)}{1-\Phi(0)} = \frac{1/\sqrt{2\pi}}{1/2} = \sqrt{\frac{2}{\pi}} \approx 0.7979$$

Por simetría, $E[Z \mid Z<0] = -0.7979$.

**Qué esperamos ver.** Las dos medias simuladas deben quedar cerca de $\pm 0.7979$.

In [1]:
import numpy as np

# Se usa una normal estándar: media 0 y desviación estándar 1.
rng = np.random.default_rng(42)
muestras = rng.normal(loc=0, scale=1, size=10_000)

negativos = muestras[muestras < 0]
positivos = muestras[muestras > 0]



print(f"Media de los negativos: {negativos.mean():.4f}")
print(f"Media de los positivos: {positivos.mean():.4f}")

Media de los negativos: -0.8003
Media de los positivos: 0.7973


**Lectura.** Salieron $-0.8003$ y $0.7973$, muy cerca del valor teórico. Lo importante para lo que sigue es la simetría: por cada valor $z$ existe un $-z$ igual de probable. La parte 2 usa justamente eso.

---

## Parte 2. Variables antitéticas en una call europea

**Datos:** $S_0 = 100$, $K = 100$, $\sigma = 0.20$, $r = 0.05$, $T = 1$. Los diez valores $Z$ los da el profesor.

**La idea.** Cada $Z$ se usa dos veces: una con $Z$ y otra con $-Z$. Después se promedian los dos payoffs. Como un $Z$ grande produce un payoff alto y su reflejo $-Z$ produce un payoff bajo (o cero), los dos errores tienden a compensarse.

**Paso 1. Precio final del activo**

$$S_T = S_0 \exp\left[\left(r - \frac{\sigma^2}{2}\right)T + \sigma\sqrt{T}\,Z\right]$$

> Este es el movimiento browniano geométrico bajo la medida neutral al riesgo. $Z$ solo mueve el exponente.

**Paso 2. Payoff de la call**

$$V = \max(S_T - K,\, 0)$$

> $V$ es una función creciente de $Z$. Por eso $V(Z)$ y $V(-Z)$ se mueven en sentidos opuestos.

**Paso 3. Promedio antitético**

$$\hat V_{\text{anti}} = \frac{V(Z) + V(-Z)}{2}$$

> Con dos simulaciones independientes la varianza del promedio sería $\operatorname{Var}(V)/2$. Con el par antitético es $\operatorname{Var}(V)\,(1+\rho)/2$, donde $\rho = \operatorname{Corr}\big(V(Z), V(-Z)\big)$. Como $V$ es monótona, $\rho \le 0$, así que el par antitético nunca es peor que dos simulaciones independientes.

In [2]:
import numpy as np
import pandas as pd

# Parámetros
S0 = 100
K = 100
sigma = 0.20
r = 0.05
T = 1

# Valores Z dados por el profesor
Z = np.array([
    0.0769,
    -0.2985,
    -0.7983,
    0.9930,
    -1.3465,
    1.2462,
    -1.1405,
    0.7615,
    0.2816,
    0.8100
])

# Precio final del activo
ST_Z = S0 * np.exp(
    (r - sigma**2 / 2) * T + sigma * np.sqrt(T) * Z
)

ST_menos_Z = S0 * np.exp(
    (r - sigma**2 / 2) * T - sigma * np.sqrt(T) * Z
)

# Payoff de la call europea
V_Z = np.maximum(ST_Z - K, 0)
V_menos_Z = np.maximum(ST_menos_Z - K, 0)

# Tabla de resultados
tabla = pd.DataFrame({
    "Z": Z,
    "S_T_Z": ST_Z,
    "Payoff_Z": V_Z,
    "S_T_-Z": ST_menos_Z,
    "Payoff_-Z": V_menos_Z,
    "Promedio_payoffs": (V_Z + V_menos_Z) / 2
})

print(tabla.round(4))
print(f"\nPayoff promedio usando Z: {V_Z.mean():.4f}")
print(f"Payoff promedio usando -Z: {V_menos_Z.mean():.4f}")

        Z     S_T_Z  Payoff_Z    S_T_-Z  Payoff_-Z  Promedio_payoffs
0  0.0769  104.6425    4.6425  101.4727     1.4727            3.0576
1 -0.2985   97.0737    0.0000  109.3846     9.3846            4.6923
2 -0.7983   87.8394    0.0000  120.8839    20.8839           10.4419
3  0.9930  125.6839   25.6839   84.4847     0.0000           12.8420
4 -1.3465   78.7179    0.0000  134.8914    34.8914           17.4457
5  1.2462  132.2125   32.2125   80.3129     0.0000           16.1062
6 -1.1405   82.0288    0.0000  129.4468    29.4468           14.7234
7  0.7615  119.9974   19.9974   88.4883     0.0000            9.9987
8  0.2816  109.0155    9.0155   97.4023     0.0000            4.5078
9  0.8100  121.1671   21.1671   87.6341     0.0000           10.5835

Payoff promedio usando Z: 11.2719
Payoff promedio usando -Z: 9.6079


**Lectura.** Fíjate en la fila con $Z = -1.3465$: con $Z$ el payoff es $0$ y con $-Z$ es $34.89$, y el promedio de ambos ($17.45$) queda en un punto intermedio. Eso es lo que estabiliza la estimación.

Con solo diez valores, la diferencia entre $11.27$ (usando $Z$) y $9.61$ (usando $-Z$) es ruido de muestreo. El promedio de las dos columnas, $\approx 10.44$, es el estimador antitético. Estos payoffs **no** están descontados; para llevarlos a valor presente se multiplican por $e^{-rT}$.

---

## Parte 3. Call asiática aritmética con variable de control geométrica

### 3.1 Qué se pide

| Dato | Valor |
|:--|:--|
| Precio inicial $S_0$ | 100 |
| Strike $K$ | 100 |
| Tasa libre de riesgo $r$ | 5% |
| Volatilidad $\sigma$ | 20% |
| Vencimiento $T$ | 1 año |
| Opción | Call asiática aritmética |
| Rutas simuladas | 100 |
| Variable de control | Call asiática geométrica |
| Fechas de observación | 12 mensuales, $t_i = i/12$ (supuesto, el enunciado no fija el número) |

Los payoffs descontados de cada ruta son:

$$X = e^{-rT}\max(\bar A - K,\,0), \qquad \bar A = \frac{1}{n}\sum_{i=1}^{n} S_{t_i}$$

$$Y = e^{-rT}\max(\bar G - K,\,0), \qquad \bar G = \left(\prod_{i=1}^{n} S_{t_i}\right)^{1/n}$$

El objetivo es estimar $E[X]$, el precio de la asiática aritmética, usando a $Y$ como variable de control.

### 3.2 De dónde viene la idea y cómo funciona

La asiática aritmética no tiene fórmula cerrada, porque una suma de lognormales no es lognormal. Pero tiene un pariente muy cercano, la asiática geométrica, cuyo precio sí se conoce exactamente.

Se simulan las dos con **las mismas trayectorias**. Si en la muestra $Y$ salió por encima de su valor verdadero, es muy probable que $X$ también haya salido alto, así que se le resta esa desviación. En fórmula:

$$\hat\theta_{CV} = \bar X - \beta\,\big(\bar Y - E[Y]\big)$$

**Paso A. Es insesgado para cualquier $\beta$.**

$$E[\hat\theta_{CV}] = E[X] - \beta\,\big(E[Y] - E[Y]\big) = E[X]$$

> No importa qué $\beta$ se elija: el término de corrección vale cero en promedio. La elección de $\beta$ solo afecta la varianza.

**Paso B. Varianza de cada observación corregida.**

$$\operatorname{Var}(X - \beta Y) = \operatorname{Var}(X) + \beta^2\operatorname{Var}(Y) - 2\beta\operatorname{Cov}(X,Y)$$

> Es una parábola en $\beta$, así que tiene un mínimo.

**Paso C. Minimizar respecto de $\beta$.**

$$\frac{d}{d\beta}: \quad 2\beta\operatorname{Var}(Y) - 2\operatorname{Cov}(X,Y) = 0 \quad\Longrightarrow\quad \beta^* = \frac{\operatorname{Cov}(X,Y)}{\operatorname{Var}(Y)}$$

> Es la pendiente de la regresión de $X$ sobre $Y$. En la práctica se estima con la covarianza y varianza muestrales.

**Paso D. Varianza resultante.**

$$\operatorname{Var}(X - \beta^* Y) = \operatorname{Var}(X) - \frac{\operatorname{Cov}(X,Y)^2}{\operatorname{Var}(Y)} = \operatorname{Var}(X)\,\big(1-\rho^2\big)$$

> Todo depende de la correlación $\rho$ entre $X$ y $Y$. Con $\rho = 0.9$ se elimina el 81% de la varianza; con $\rho = 0.99$, el 98%.

### 3.3 Por qué la geométrica es una buena variable de control

1. **Se conoce su precio exacto.** El logaritmo del promedio geométrico es el promedio de logaritmos normales, así que es normal y $\bar G$ es lognormal. Eso da una fórmula tipo Black-Scholes.
2. **Está muy correlacionada con la aritmética.** Siempre se cumple $\bar A \ge \bar G$ (desigualdad entre medias aritmética y geométrica) y, con volatilidades normales, los dos promedios son casi iguales.

### 3.4 Cálculo paso a paso

**Paso 1. Simular las 100 rutas.** Con $\Delta t = T/n = 1/12$ y $Z_i \sim N(0,1)$ independientes:

$$S_{t_i} = S_{t_{i-1}}\exp\left[\left(r - \frac{\sigma^2}{2}\right)\Delta t + \sigma\sqrt{\Delta t}\,Z_i\right]$$

> Esta discretización es exacta para el movimiento browniano geométrico, no una aproximación de Euler. Con 12 fechas no hay error de discretización.

**Paso 2. Promedios y payoffs.** Para cada ruta se calculan $\bar A$ y $\bar G$, y de ahí $X$ y $Y$ (ya descontados con $e^{-rT}$).

**Paso 3. Precio exacto de la geométrica, $E[Y]$.** Como $\ln \bar G$ es normal:

$$\ln \bar G = \frac{1}{n}\sum_{i=1}^{n}\ln S_{t_i} \sim N\big(\mu_G,\ \sigma_G^2\big)$$

$$\mu_G = \ln S_0 + \left(r - \frac{\sigma^2}{2}\right)\bar t, \qquad \bar t = \frac{1}{n}\sum_{i=1}^{n} t_i$$

$$\sigma_G^2 = \frac{\sigma^2}{n^2}\sum_{i=1}^{n}\sum_{j=1}^{n}\min(t_i, t_j)$$

> El $\min(t_i, t_j)$ aparece porque la covarianza del movimiento browniano es $\operatorname{Cov}(W_{t_i}, W_{t_j}) = \min(t_i, t_j)$.

Con los datos del problema: $\bar t = 0.5417$, $\mu_G = 4.6214$ y $\sigma_G^2 = 0.015046$ (es decir, $\sigma_G = 0.1227$).

Y el precio de una call sobre una variable lognormal:

$$E[Y] = e^{-rT}\Big[e^{\mu_G + \sigma_G^2/2}\,N(d_1) - K\,N(d_2)\Big]$$

$$d_1 = \frac{\mu_G - \ln K + \sigma_G^2}{\sigma_G}, \qquad d_2 = d_1 - \sigma_G$$

> Es la fórmula de Black-Scholes escrita con la media de $\bar G$, que es $e^{\mu_G+\sigma_G^2/2}$, en lugar del forward. Aquí $d_1 = 0.2551$, $d_2 = 0.1325$ y $E[Y] = 5.9402$.

**Paso 4. Estimar $\beta$ con la muestra.**

$$\hat\beta = \frac{\widehat{\operatorname{Cov}}(X, Y)}{\widehat{\operatorname{Var}}(Y)}$$

> Como $X$ y $Y$ son casi la misma variable, se espera $\hat\beta$ cerca de 1.

**Paso 5. Estimador con control y su error estándar.**

$$\hat\theta_{CV} = \bar X - \hat\beta\,\big(\bar Y - E[Y]\big), \qquad EE = \frac{s\,(X - \hat\beta Y)}{\sqrt{N}}$$

> El error estándar es el de la variable combinada $X - \hat\beta Y$, no el de $X$. Ahí es donde se nota la reducción de varianza.

### 3.5 Implementación

In [3]:
# Ejercicio: call asiatica aritmetica con variable de control geometrica
# Se usan 100 rutas y 12 observaciones mensuales (t = 1/12, ..., 1).
import math
import numpy as np
import pandas as pd

S0 = 100
K = 100
sigma = 0.20
r = 0.05
T = 1
n_rutas = 100
n_observaciones = 12

rng = np.random.default_rng(42)
dt = T / n_observaciones
Z = rng.standard_normal((n_rutas, n_observaciones))
log_retornos = (r - sigma**2 / 2) * dt + sigma * np.sqrt(dt) * Z
precios = S0 * np.exp(np.cumsum(log_retornos, axis=1))

promedio_aritmetico = precios.mean(axis=1)
promedio_geometrico = np.exp(np.log(precios).mean(axis=1))
descuento = np.exp(-r * T)
payoff_aritmetico = descuento * np.maximum(promedio_aritmetico - K, 0)
payoff_geometrico = descuento * np.maximum(promedio_geometrico - K, 0)

# El promedio geometrico tiene lognormalidad; se usa su precio exacto como E[Y].
tiempos = dt * np.arange(1, n_observaciones + 1)
media_log_geometrica = np.log(S0) + (r - sigma**2 / 2) * tiempos.mean()
var_log_geometrica = sigma**2 * np.minimum.outer(tiempos, tiempos).sum() / n_observaciones**2
sd_log_geometrica = np.sqrt(var_log_geometrica)
d1 = (media_log_geometrica - np.log(K) + var_log_geometrica) / sd_log_geometrica
d2 = d1 - sd_log_geometrica
N = lambda x: 0.5 * (1 + math.erf(x / np.sqrt(2)))
precio_geometrico_exacto = np.exp(-r * T) * (
    np.exp(media_log_geometrica + var_log_geometrica / 2) * N(d1) - K * N(d2)
)

# Estimador de control: X_bar - beta * (Y_bar - E[Y]).
beta = np.cov(payoff_aritmetico, payoff_geometrico, ddof=1)[0, 1] / np.var(payoff_geometrico, ddof=1)
estimacion_sin_control = payoff_aritmetico.mean()
estimacion_con_control = estimacion_sin_control - beta * (payoff_geometrico.mean() - precio_geometrico_exacto)
error_estandar_sin_control = payoff_aritmetico.std(ddof=1) / np.sqrt(n_rutas)
error_estandar_con_control = (payoff_aritmetico - beta * payoff_geometrico).std(ddof=1) / np.sqrt(n_rutas)

print(f"Precio geometrico exacto (control): {precio_geometrico_exacto:.4f}")
print(f"Beta estimada: {beta:.4f}")
print(f"Call asiatica aritmetica sin control: {estimacion_sin_control:.4f} (EE: {error_estandar_sin_control:.4f})")
print(f"Call asiatica aritmetica con control: {estimacion_con_control:.4f} (EE: {error_estandar_con_control:.4f})")
correlacion = np.corrcoef(payoff_aritmetico, payoff_geometrico)[0, 1]
factor_varianza = (error_estandar_sin_control / error_estandar_con_control) ** 2
print(f"Correlacion entre payoffs: {correlacion:.4f}")
print(f"Reduccion de varianza: {factor_varianza:.0f} veces")

resultados = pd.DataFrame({
    "Ruta": np.arange(1, n_rutas + 1),
    "Promedio aritmetico": promedio_aritmetico,
    "Promedio geometrico": promedio_geometrico,
    "Payoff aritmetico descontado": payoff_aritmetico,
    "Payoff geometrico descontado": payoff_geometrico,
})
print("\nPrimeras 10 de las 100 rutas simuladas:")
print(resultados.head(10).round(4).to_string(index=False))


Precio geometrico exacto (control): 5.9402
Beta estimada: 1.0362
Call asiatica aritmetica sin control: 5.3316 (EE: 0.7744)
Call asiatica aritmetica con control: 6.1847 (EE: 0.0202)
Correlacion entre payoffs: 0.9997
Reduccion de varianza: 1476 veces

Primeras 10 de las 100 rutas simuladas:
 Ruta  Promedio aritmetico  Promedio geometrico  Payoff aritmetico descontado  Payoff geometrico descontado
    1              93.6752              93.4575                        0.0000                        0.0000
    2             107.1607             107.1111                        6.8114                        6.7642
    3             109.4282             109.0370                        8.9684                        8.5963
    4             100.3204             100.1849                        0.3047                        0.1759
    5             101.8342             101.7226                        1.7448                        1.6386
    6              96.5600              96.4166               

### 3.6 Resultados y lectura

| Estimador | Valor | Error estándar |
|:--|:--:|:--:|
| Asiática aritmética sin control | 5.3316 | 0.7744 |
| Asiática aritmética con control | 6.1847 | 0.0202 |
| Asiática geométrica exacta $E[Y]$ | 5.9402 | no aplica |

**Cómo actuó la corrección.** En esta muestra las rutas salieron bajas: el promedio muestral de $Y$ fue $5.1169$, por debajo del valor exacto $5.9402$. La corrección es

$$\hat\beta\,(\bar Y - E[Y]) = 1.0362 \times (5.1169 - 5.9402) \approx -0.853$$

y al restarla, $5.3316 - (-0.853) = 6.1847$. Es decir, la geométrica detectó que la muestra venía baja y empujó la estimación de la aritmética hacia arriba.

**Reducción de varianza.** El error estándar baja de $0.7744$ a $0.0202$, unas 38 veces menos (alrededor de 1,500 veces menos varianza). Esto concuerda con la fórmula $1/(1-\rho^2)$ y la correlación de $0.9997$ que imprime la celda.

**Chequeos de coherencia.**

- La aritmética con control ($6.18$) queda por encima de la geométrica exacta ($5.94$), como exige $\bar A \ge \bar G$.
- Una simulación aparte con 2 millones de rutas, hecha para validar y no incluida en este cuaderno, da alrededor de $6.16$ para la aritmética. El resultado con control ($6.18 \pm 0.02$) es consistente con ese valor, mientras que el de sin control ($5.33$) queda aproximadamente un error estándar por debajo, que es simple ruido de muestreo con solo 100 rutas.
- Los números dependen de la semilla (`default_rng(42)`). Con otra semilla cambian, pero el error estándar con control seguirá siendo pequeño.

---

## Diccionario de términos

| Término | Significado |
|:--|:--|
| **Payoff** | Lo que paga la opción al vencimiento. En una call, $\max(S_T - K, 0)$. |
| **Call europea** | Opción de compra que solo se puede ejercer al vencimiento. |
| **Call asiática** | Opción cuyo payoff depende del promedio del precio durante la vida de la opción y no solo del precio final. |
| **Promedio aritmético** | $\frac{1}{n}\sum S_{t_i}$. Es el que usa la asiática aritmética. |
| **Promedio geométrico** | $\left(\prod S_{t_i}\right)^{1/n}$. Siempre es menor o igual que el aritmético. |
| **Strike ($K$)** | Precio pactado al que se compara el activo para calcular el payoff. |
| **Movimiento browniano geométrico** | Modelo donde $\ln S_t$ se mueve como un browniano con deriva. Es el modelo de Black-Scholes. |
| **Medida neutral al riesgo** | Forma de valuar donde el activo crece a la tasa libre de riesgo $r$ y el precio es el valor esperado descontado del payoff. |
| **Lognormal** | Variable cuyo logaritmo es normal. |
| **Discretización exacta** | Simular el proceso en fechas discretas usando su distribución verdadera, sin error de aproximación. |
| **Ruta o trayectoria** | Una simulación completa del precio a lo largo de las fechas de observación. |
| **Estimador insesgado** | Estimador cuyo valor esperado es el valor verdadero. |
| **Error estándar (EE)** | Desviación estándar del estimador, $s/\sqrt{N}$. Mide cuánto fluctuaría la estimación entre repeticiones. |
| **Reducción de varianza** | Conjunto de técnicas que bajan el error estándar sin aumentar el número de simulaciones. |
| **Variable antitética** | Pareja $(Z, -Z)$ que se promedia para que los errores se compensen. |
| **Variable de control** | Variable $Y$ correlacionada con $X$ y de media conocida, usada para corregir la estimación de $E[X]$. |
| **$\beta$ óptimo** | Coeficiente $\operatorname{Cov}(X,Y)/\operatorname{Var}(Y)$ que minimiza la varianza del estimador con control. |
| **Correlación ($\rho$)** | Medida entre $-1$ y $1$ de qué tanto se mueven juntas dos variables. |
| **Semilla (seed)** | Número que fija el generador aleatorio para que los resultados se puedan repetir. |